# Locality-restricted chiral index for square-QDM cages

This notebook tests whether the eight compact IPR cages form regional chiral kernels and whether the ninth collective-cancellation cage belongs to their direct span.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "qlinks").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate repository root")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qlinks.caging import (
    CageSearchConfig, CageSearcher,
    diagnose_locality_restricted_chiral_profile,
    regional_chiral_kernel_span,
)
from qlinks.models import SquareQDMModel

In [ ]:
model = SquareQDMModel(
    lx=4, ly=4, boundary_condition="periodic",
    winding_x=0, winding_y=0, winding_convention="electric",
    coup_kin=1.0, coup_pot=1.0,
)
build = model.build(basis_solver="dfs", builder="sparse", backend="scipy", sort_basis=True)
search = CageSearcher.from_model_build_result(
    build,
    config=CageSearchConfig(
        search_type="type1", tolerance=1e-10,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=64, ipr_candidate_count=32, ipr_random_seed=1234,
    ),
).run()
records = tuple(search[(0, 4)])

def full_state(cage_state):
    vector = np.zeros(search.hilbert_size, dtype=np.complex128)
    vector[np.asarray(cage_state.support, dtype=np.int64)] = cage_state.local_state
    return vector

states = np.column_stack([full_state(record.cage_state) for record in records])
regions = tuple(record.cage_state.support for record in records[:8])
len(records), tuple(len(region) for region in regions)

In [ ]:
span_report = regional_chiral_kernel_span(
    build.kinetic, regions, states, tolerance=1e-10
)
pd.Series(span_report.to_summary_dict())

In [ ]:
rows = []
for index, record in enumerate(records):
    profile = diagnose_locality_restricted_chiral_profile(
        build.kinetic, regions,
        target_state=full_state(record.cage_state),
        tolerance=1e-10,
    )
    rows.append({
        "record": index,
        "support_size": record.cage_state.support_size,
        "regional_zero_modes": profile.n_regional_target_zero_modes,
        "uncovered_weight": profile.uncovered_target_weight,
        "regional_weights": tuple(round(entry.target_weight or 0.0, 8) for entry in profile.entries),
    })
pd.DataFrame(rows)

In [ ]:
local_index_table = pd.DataFrame(
    diagnose_locality_restricted_chiral_profile(
        build.kinetic, regions, tolerance=1e-10
    ).entries[i].to_summary_dict()
    for i in range(len(regions))
)
local_index_table

## Interpretation

Each compact region has one right-kernel cage mode, but its local chiral index is negative: the cage mode is paired rather than protected by sublattice imbalance. The eight regional kernels nevertheless span exactly eight dimensions of the nine-dimensional cage manifold. The ninth state is not a sum of independently closed regional zero modes; it requires collective interference across the larger support. Thus the useful invariant here is a **locality-restricted kernel decomposition**, not the ordinary chiral index alone.